# Exploratory Data Analysis Execution

Part B: descriptive distributions and relationships, before decision-tree modeling. Saved text outputs are actual Python executions using the sequential runner, not a Jupyter kernel. No cleaned values are modified. Figures are separately generated by `python3 scripts/eda.py` with matplotlib installed.

## 1. Load validated data and calculate within-group rates

Capture the code and output together. The denominator is the number of customers in each contract category.


In [1]:
from pathlib import Path
import csv, json, subprocess, sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/processed/telco_clean.csv").exists())
sys.path.insert(0, str(ROOT / "scripts"))
from eda import load_data
rows, original = load_data()
print(f"Loaded {len(rows):,} customers; input checksum verified.")
print("Contract          Yes / group total    Churn share")
for category in ["Month-to-month", "One year", "Two year"]:
    group = [r for r in rows if r["Contract"] == category]
    yes = sum(r["Churn"] == "Yes" for r in group)
    print(f"{category:17} {yes:4} / {len(group):4}          {yes/len(group):.2%}")


Loaded 7,043 customers; input checksum verified.
Contract          Yes / group total    Churn share
Month-to-month    1655 / 3875          42.71%
One year           166 / 1473          11.27%
Two year            48 / 1695          2.83%


## 2. Check missing-value denominators and numeric relationships

Keep all customers in the sample. Exclude missing charges only when those charges are required for a calculation. This is not imputation or global row deletion.


In [2]:
import statistics as st
for field in ["tenure", "MonthlyCharges", "TotalCharges"]:
    for label in ["No", "Yes"]:
        group = [r for r in rows if r["Churn"] == label]
        values = [float(r[field]) for r in group if r[field] != ""]
        print(f"{field:15} Churn={label:3} valid={len(values):4} "
              f"missing={len(group)-len(values):2} median={st.median(values):.2f}")
pairs = [(float(r["tenure"]), float(r["TotalCharges"]))
         for r in rows if r["TotalCharges"] != ""]
x, y = zip(*pairs)
print(f"Tenure vs TotalCharges: Pearson r={st.correlation(x,y):.3f}, n={len(pairs):,}")


tenure          Churn=No  valid=5174 missing= 0 median=38.00
tenure          Churn=Yes valid=1869 missing= 0 median=10.00
MonthlyCharges  Churn=No  valid=5174 missing= 0 median=64.43
MonthlyCharges  Churn=Yes valid=1869 missing= 0 median=79.65
TotalCharges    Churn=No  valid=5163 missing=11 median=1683.60
TotalCharges    Churn=Yes valid=1869 missing= 0 median=703.55
Tenure vs TotalCharges: Pearson r=0.826, n=7,032


## 3. Generate tables and verify denominators

The verification independently regroups the data and recalculates Pearson r from its centered-product formula. Each category field must reconcile to the full sample. Capture the command and verified output.


In [3]:
run = subprocess.run([sys.executable, str(ROOT / "scripts/eda.py"), "--tables-only"],
                     capture_output=True, text=True, check=True)
print(run.stdout, end="")
check = subprocess.run([sys.executable, str(ROOT / "scripts/verify_eda.py")],
                       capture_output=True, text=True, check=True)
print(check.stdout, end="")
assert (ROOT / "data/processed/telco_clean.csv").read_bytes() == original


EDA COMPLETE | 7,043 customers | 43 category rows | 9 numeric summaries
Month-to-month: 1655/3875 = 42.71%
One year: 166/1473 = 11.27%
Two year: 48/1695 = 2.83%
Numeric correlations: [('tenure', 'MonthlyCharges', 0.248, 7043), ('tenure', 'TotalCharges', 0.826, 7032), ('MonthlyCharges', 'TotalCharges', 0.651, 7032)]
EDA COMPLETE | 7,043 customers | 43 category rows | 9 numeric summaries
Month-to-month: 1655/3875 = 42.71%
One year: 166/1473 = 11.27%
Two year: 48/1695 = 2.83%
Numeric correlations: [('tenure', 'MonthlyCharges', 0.248, 7043), ('tenure', 'TotalCharges', 0.826, 7032), ('MonthlyCharges', 'TotalCharges', 0.651, 7032)]
VERIFIED: each category field reconciles to 7,043 customers and 1,869 Yes labels.
VERIFIED: means, medians, missing denominators and independent Pearson calculations agree.
VERIFIED: all EDA tables reproduce byte-for-byte; cleaned input is unchanged.


## 4. Generated figures

These images are the saved outputs of the plotting script, not independent model results. For process evidence, capture the calculation cells above; use PNG or SVG files directly for report/slides.

![Contract and internet service](../reports/eda/figures/04_contract_internet_churn.png)

![Numeric distributions by churn](../reports/eda/figures/03_numeric_by_churn.png)

![Numeric correlations](../reports/eda/figures/06_numeric_correlations.png)

## Interpretation boundary

These are full-sample associations, not causal effects, customer segments or held-out decision-tree evaluation. C owns training, validation and test evaluation. D owns final customer profiles and business insights. The high tenure–TotalCharges correlation does not automatically justify removing either feature from a tree.
